<a href="https://colab.research.google.com/github/janhaviojha/Swiggy-recommendation-system-learn/blob/main/reccomendation-system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(42)

In [20]:
N_USERS, N_REST, N_SESSIONS, N_CAND, N_DAYS = 500, 100, 5000, 20, 60
CUISINES = ["north_indian", "biryani", "chinese", "south_indian", "snacks"]


In [21]:
users = pd.DataFrame({
    "user_id": range(N_USERS),
    "budget": rng.normal(350, 120, N_USERS).clip(100, 900),   # hidden taste
    "fav_cuisine": rng.choice(CUISINES, N_USERS),
    "is_veg_user": rng.random(N_USERS) < 0.3,
})
rests = pd.DataFrame({
    "rest_id": range(N_REST),
    "rating": rng.normal(4.0, 0.5, N_REST).clip(2.5, 5.0),
    "price": rng.normal(350, 150, N_REST).clip(80, 1000),
    "cuisine": rng.choice(CUISINES, N_REST),
    "is_veg_rest": rng.random(N_REST) < 0.25,
})

In [22]:
print(users.head())
print(rests.head())


   user_id      budget   fav_cuisine  is_veg_user
0        0  386.566050       chinese        False
1        1  225.201907       biryani        False
2        2  440.054143  south_indian        False
3        3  462.867766  north_indian         True
4        4  115.875777       biryani        False
   rest_id    rating       price       cuisine  is_veg_rest
0        0  4.043253  390.514315       chinese        False
1        1  4.163505  411.795737       chinese        False
2        2  3.603737  628.924221       chinese         True
3        3  4.013293  420.282917       chinese        False
4        4  4.285269  448.683202  north_indian        False


In [25]:
rows = []
for s in range(N_SESSIONS):
    u = users.iloc[rng.integers(N_USERS)]
    day, hour = rng.integers(N_DAYS), rng.choice([12, 13, 19, 20, 21, 22])
    rainy = int(rng.random() < 0.3)
    cand = rests.sample(N_CAND, random_state=int(rng.integers(1e9)))
    for _, r in cand.iterrows():
        dist = rng.uniform(0.5, 10)
        # Hidden "true" behaviour the model must discover
        logit = (-2.0 + 0.9 * (r.rating - 4.0) - 0.35 * dist
                 - 0.004 * abs(r.price - u.budget)
                 + 1.2 * (r.cuisine == u.fav_cuisine)
                 - 3.0 * (u.is_veg_user and not r.is_veg_rest)
                 + 1.0 * (rainy and r.cuisine == "snacks"))
        ordered = int(rng.random() < 1 / (1 + np.exp(-logit)))
        rows.append((s, day, hour, rainy, u.user_id, r.rest_id, dist, ordered))

df = pd.DataFrame(rows, columns=["session", "day", "hour", "is_rainy",
                                 "user_id", "rest_id", "distance_km", "ordered"])
df = df.merge(users, on="user_id").merge(rests, on="rest_id")
print(f"Rows: {len(df):,} | order rate: {df.ordered.mean():.2%}")

Rows: 100,000 | order rate: 2.41%


In [26]:
train = df[df.day < 40].copy()
val = df[(df.day >= 40) & (df.day < 50)].copy()
test = df[df.day >= 50].copy()


In [27]:
hist = train[train.ordered == 1]
user_avg_value = hist.groupby("user_id").price.mean()
user_top_cuisine = hist.groupby("user_id").cuisine.agg(lambda x: x.mode()[0])
global_avg = hist.price.mean()

In [28]:
hist = train[train.ordered == 1]
user_avg_value = hist.groupby("user_id").price.mean()
user_top_cuisine = hist.groupby("user_id").cuisine.agg(lambda x: x.mode()[0])
global_avg = hist.price.mean()

In [29]:
def add_features(d):
    d = d.copy()
    d["user_avg_value"] = d.user_id.map(user_avg_value).fillna(global_avg)
    d["price_gap"] = (d.price - d.user_avg_value).abs()
    d["cuisine_match"] = (d.cuisine == d.user_id.map(user_top_cuisine)).astype(int)
    d["diet_clash"] = (d.is_veg_user & ~d.is_veg_rest).astype(int)
    d["is_snacks"] = (d.cuisine == "snacks").astype(int)
    d["rain_x_snacks"] = d.is_rainy * d.is_snacks
    d["is_veg_rest"] = d.is_veg_rest.astype(int)
    return d

In [30]:
FEATURES = ["rating", "distance_km", "price", "price_gap", "cuisine_match",
            "diet_clash", "is_veg_rest", "hour", "is_rainy", "rain_x_snacks"]
train, val, test = map(add_features, (train, val, test))

In [31]:
def ndcg_at_k(df_, score_col, k=10):
    scores = []
    for _, g in df_.groupby("session"):
        if g.ordered.sum() == 0:
            continue                      # no relevant item, skip
        top = g.sort_values(score_col, ascending=False).ordered.values[:k]
        dcg = (top / np.log2(np.arange(2, len(top) + 2))).sum()
        ideal = np.sort(g.ordered.values)[::-1][:k]
        idcg = (ideal / np.log2(np.arange(2, len(ideal) + 2))).sum()
        scores.append(dcg / idcg)
    return np.mean(scores)

In [32]:
for d in (val, test):
    d["baseline"] = d.rating - 0.3 * d.distance_km     # no ML: "good + near"


In [33]:
print("train rows:", len(train), "| orders in train:", train.ordered.sum())
print(train[FEATURES].isna().sum())

train rows: 65280 | orders in train: 1560
rating           0
distance_km      0
price            0
price_gap        0
cuisine_match    0
diet_clash       0
is_veg_rest      0
hour             0
is_rainy         0
rain_x_snacks    0
dtype: int64


In [34]:
logreg = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=500))
logreg.fit(train[FEATURES], train.ordered)



Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression', LogisticRegression(max_iter=500))])

In [35]:
gbm = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.1, max_iter=200)
gbm.fit(train[FEATURES], train.ordered)

HistGradientBoostingClassifier(max_depth=4, max_iter=200)

In [36]:
for d in (val, test):
    d["logreg"] = logreg.predict_proba(d[FEATURES])[:, 1]
    d["gbm"] = gbm.predict_proba(d[FEATURES])[:, 1]


In [37]:
print("\nNDCG@10          validation   test")
for name in ["baseline", "logreg", "gbm"]:
    print(f"{name:<16} {ndcg_at_k(val, name):.3f}        {ndcg_at_k(test, name):.3f}")


NDCG@10          validation   test
baseline         0.524        0.531
logreg           0.531        0.522
gbm              0.513        0.516


In [38]:
coefs = pd.Series(logreg[-1].coef_[0], index=FEATURES).sort_values()
print("\nLearned weights (features standardized, so comparable):")
print(coefs.round(2).to_string())
print(f"Bias (intercept): {logreg[-1].intercept_[0]:.2f}")



Learned weights (features standardized, so comparable):
diet_clash      -1.10
distance_km     -0.86
price_gap       -0.70
price           -0.04
hour            -0.02
is_veg_rest     -0.01
is_rainy         0.01
rain_x_snacks    0.20
rating           0.43
cuisine_match    0.67
Bias (intercept): -5.00


In [39]:
one = test[test.session == test.session.iloc[0]]
ranked = one.sort_values("logreg", ascending=False)

print(ranked[["rest_id", "cuisine", "rating", "distance_km", "price",
              "logreg", "ordered"]].head(10).to_string(index=False))

 rest_id      cuisine   rating  distance_km      price   logreg  ordered
      26      chinese 3.758884     2.536202 311.905598 0.146342        0
      11      chinese 4.522436     2.843395 195.140447 0.113591        0
      30      chinese 3.776421     6.290748 479.300008 0.033972        0
      75 north_indian 2.611003     0.593713 348.680797 0.027835        0
       0      chinese 4.043253     9.912937 390.514315 0.025571        0
       5      chinese 4.309095     2.360601 756.611829 0.024290        0
      24 south_indian 3.980731     4.182694 273.934208 0.016696        0
      23 south_indian 4.387247     5.724624 293.568184 0.015720        0
      77 north_indian 3.410000     4.063580 287.172688 0.011987        0
      92      biryani 4.483733     5.911659 213.283457 0.009445        0
